In [1]:
import os
import glob
from collections import namedtuple

# 1) Disable XNNPACK BEFORE importing TF (debug only)
os.environ["TF_TFLITE_USE_XNNPACK"] = "0"

import numpy as np
import matplotlib.pyplot as plt
import cv2
from tflite_inference_debug import TFLiteInference
from tqdm import tqdm

%load_ext autoreload
%autoreload 2

2026-01-13 15:34:02.598338: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-13 15:34:02.615814: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-13 15:34:02.621002: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-13 15:34:02.634526: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-13 15:34:03.518706: W tensorflow/compiler/tf2

In [2]:
ModelParams = namedtuple("ModelParams", ["model_name", "int8_path", "fp32_path", "data_path"])

In [3]:
IMGSZ = 320
CALIBTARION_N = 100

COLLECTED_OUTPUTS_DIR = "collected_outputs"

BASE_RESULTS_PTH = "/home/klymy/projects/researches/people_detection_pg2/results"
BASE_DATA_PTH = "/home/klymy/data/cam4_revB/yolo_dataset"

# parameters = [
#     ModelParams(
#         model_name="cam3_revB_batch1_11_margin_15_rq_wb_alif_clahe_box_01_dfl_01_2nd",
#         int8_path=os.path.join(BASE_RESULTS_PTH, "cam3_revB_batch1_11_margin_15_rq_wb_alif_clahe_box_01_dfl_01_2nd", "weights/best_non_separated_saved_model/best_full_integer_quant.tflite"),
#         fp32_path=os.path.join(BASE_RESULTS_PTH, "cam3_revB_batch1_11_margin_15_rq_wb_alif_clahe_box_01_dfl_01_2nd", "weights/best_non_separated_saved_model/best_float32.tflite"),
#         data_path=os.path.join(BASE_DATA_PTH, "batch1_11_margin_15_bg_wb_alif_clahe/images"),
#     ),
#     ModelParams(
#         model_name="cam3_revB_batch1_11_margin_15_rq_wb_box_01_dfl_01_2nd",
#         int8_path=os.path.join(BASE_RESULTS_PTH, "cam3_revB_batch1_11_margin_15_rq_wb_box_01_dfl_01_2nd", "weights/best_non_separated_saved_model/best_full_integer_quant.tflite"),
#         fp32_path=os.path.join(BASE_RESULTS_PTH, "cam3_revB_batch1_11_margin_15_rq_wb_box_01_dfl_01_2nd", "weights/best_non_separated_saved_model/best_float32.tflite"),
#         data_path=os.path.join(BASE_DATA_PTH, "batch1_11_margin_15_bg_wb_alif/images"),
#     ),
# ]

os.makedirs(COLLECTED_OUTPUTS_DIR, exist_ok=True)

parameters = [
    ModelParams(
        model_name="cam3_revB_batch1_11_margin_15_rq_yolov8n_wb_alif_clahe_box_01_dfl_01_2nd",
        int8_path=os.path.join(BASE_RESULTS_PTH, "cam3_revB_batch1_11_margin_15_rq_yolov8n_wb_alif_clahe_box_01_dfl_01_2nd", "weights/best_non_separated_saved_model/best_full_integer_quant.tflite"),
        fp32_path=os.path.join(BASE_RESULTS_PTH, "cam3_revB_batch1_11_margin_15_rq_yolov8n_wb_alif_clahe_box_01_dfl_01_2nd", "weights/best_non_separated_saved_model/best_float32.tflite"),
        data_path=os.path.join(BASE_DATA_PTH, "batch1_11_margin_15_bg_wb_alif_clahe/images"),
    ),
    ModelParams(
        model_name="cam3_revB_batch1_11_margin_15_yolov8n_wb_alif_box_01_dfl_01_2nd",
        int8_path=os.path.join(BASE_RESULTS_PTH, "cam3_revB_batch1_11_margin_15_yolov8n_wb_alif_box_01_dfl_01_2nd", "weights/best_non_separated_saved_model/best_full_integer_quant.tflite"),
        fp32_path=os.path.join(BASE_RESULTS_PTH, "cam3_revB_batch1_11_margin_15_yolov8n_wb_alif_box_01_dfl_01_2nd", "weights/best_non_separated_saved_model/best_float32.tflite"),
        data_path=os.path.join(BASE_DATA_PTH, "batch1_11_margin_15_bg_wb_alif/images"),
    ),
]

In [4]:
def load_data(base_pth: str, calibration_n: int):
    # Load and run inference on first image
    train_img_pths = glob.glob(os.path.join(base_pth, "train/*.jpg"))
    print(f"Found {len(train_img_pths)} images")

    val_img_pths = glob.glob(os.path.join(base_pth, "val/*.jpg"))
    print(f"Found {len(val_img_pths)} images")

    test_img_pths = glob.glob(os.path.join(base_pth, "holdout/*.jpg"))
    print(f"Found {len(test_img_pths)} images")

    # img = cv2.imread(img_pths[0])
    # img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    np.random.seed(42)
    np.random.shuffle(train_img_pths)
    np.random.shuffle(val_img_pths)
    np.random.shuffle(test_img_pths)
    img_pths = train_img_pths[:calibration_n] + val_img_pths[:calibration_n] + test_img_pths[:calibration_n]

    print(f"Total images for calibration: {len(img_pths)}")

    return img_pths

In [5]:
def collect_data(img_pths: list, fp32_inference: TFLiteInference, uint8_inference: TFLiteInference, folder_name: str):
    buf_fp32_boxes = []
    buf_fp32_scores = []
    buf_uint8_boxes = []
    buf_uint8_scores = []

    for img_pth in tqdm(img_pths):
        img = cv2.imread(img_pth)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        fp32_output = fp32_inference.infer_image(img.astype(np.float32) / 255)
        uint8_output = uint8_inference.infer_image(img.astype(np.uint8))
        
        uint8_scores = uint8_output["outputs"][0]
        fp32_scores = fp32_output["outputs"][1]

        uint8_coordinates = uint8_output["outputs"][1]
        fp32_coordinates = fp32_output["outputs"][0]

        idxs = np.where(uint8_scores[0] > 0.2)[0]
        buf_fp32_scores.append(fp32_scores[0, idxs])
        buf_fp32_boxes.append(fp32_coordinates[0, idxs])
        buf_uint8_scores.append(uint8_scores[0, idxs])
        buf_uint8_boxes.append(uint8_coordinates[0, idxs])


    buf_fp32_scores = np.concatenate(buf_fp32_scores)
    buf_fp32_boxes = np.concatenate(buf_fp32_boxes)
    buf_uint8_scores = np.concatenate(buf_uint8_scores)
    buf_uint8_boxes = np.concatenate(buf_uint8_boxes)

    print("FP32 scores shape:", buf_fp32_scores.shape)
    print("FP32 boxes shape:", buf_fp32_boxes.shape)
    print("UINT8 scores shape:", buf_uint8_scores.shape)
    print("UINT8 boxes shape:", buf_uint8_boxes.shape)

    os.makedirs(folder_name, exist_ok=True)
    np.save(f"{folder_name}/fp32_scores.npy", buf_fp32_scores)
    np.save(f"{folder_name}/fp32_boxes.npy", buf_fp32_boxes)
    np.save(f"{folder_name}/uint8_scores.npy", buf_uint8_scores)
    np.save(f"{folder_name}/uint8_boxes.npy", buf_uint8_boxes)

In [6]:
for params in parameters:
    print(f"Processing model: {params.model_name}")
    img_pths = load_data(params.data_path, CALIBTARION_N)
    
    layer_indices = []  # Add layer indices here if needed, e.g., [100, 200, 300]

    fp32_inference = TFLiteInference(params.fp32_path, is_uint8=False, imgsz=IMGSZ, layer_indices=layer_indices)
    uint8_inference = TFLiteInference(params.int8_path, is_uint8=True, imgsz=IMGSZ, layer_indices=layer_indices)

    folder_name = os.path.join(COLLECTED_OUTPUTS_DIR, params.model_name)
    collect_data(img_pths, fp32_inference, uint8_inference, folder_name)
    print("=============================================\n")

Processing model: cam3_revB_batch1_11_margin_15_rq_yolov8n_wb_alif_clahe_box_01_dfl_01_2nd
Found 13311 images
Found 1474 images
Found 148 images
Total images for calibration: 300


  0%|          | 0/300 [00:00<?, ?it/s]

100%|██████████| 300/300 [05:25<00:00,  1.09s/it]


FP32 scores shape: (9525, 1)
FP32 boxes shape: (9525, 6)
UINT8 scores shape: (9525, 1)
UINT8 boxes shape: (9525, 6)

Processing model: cam3_revB_batch1_11_margin_15_yolov8n_wb_alif_box_01_dfl_01_2nd
Found 13311 images
Found 1474 images
Found 148 images
Total images for calibration: 300


100%|██████████| 300/300 [05:26<00:00,  1.09s/it]

FP32 scores shape: (9490, 1)
FP32 boxes shape: (9490, 6)
UINT8 scores shape: (9490, 1)
UINT8 boxes shape: (9490, 6)

